In [1]:
import math
from enum import Enum

class LayerType(Enum):  # Вспомогательное перечисление для типов слоев
    POINT = "point"
    CENTER_POINT = "center_point"
    CIRCLE_POINT = "circle_point"
    GENERIC_POINT = "generic_point"
    GAME_CELL = "game_cell"
    LOGISTIC_MAP_POINT = "logistic_map_point"
    MANDELBROT_POINT = "mandelbrot_point"
    SQUARE_POINT = "square_point" # Добавлен для "невозможного квадрата"

class Layer:
    def __init__(self, name, layer_type, x=None, y=None, z=None, state=None, row=None, col=None, is_in_set=None, iteration_value=None):
        self.name = name
        self.type = layer_type # Используем Enum LayerType
        self.x = x  # math.numeral - в Python используем float или int
        self.y = y  # math.numeral - в Python используем float или int
        self.z = z  # math.numeral - в Python используем float или int
        self.state = state # enums.CellState - для крестиков-ноликов, Enum CellState
        self.row = row # integer - для крестиков-ноликов, int
        self.col = col # integer - для крестиков-ноликов, int
        self.is_in_set = is_in_set # boolean - для множества Мандельброта, bool
        self.iteration_value = iteration_value # list<math.numeral> - для логистической карты, list

    def __repr__(self): # Для удобного вывода объектов Layer
        return f"Layer(name='{self.name}', type='{self.type.value}', x={self.x}, y={self.y}, z={self.z}, state={self.state}, row={self.row}, col={self.col}, is_in_set={self.is_in_set}, iteration_value={self.iteration_value})"

In [2]:
class Figure:
    def __init__(self, name, figure_type):
        self.name = name
        self.type = figure_type #  тип фигуры (пока string, можно сделать Enum в будущем)
        self.center = None # layers(type=center_point) - будет объектом Layer типа center_point
        self.radius = None # math.numeral - float или int
        self.side_length = None # math.numeral - float или int
        self.circle_points = [] # continuum{...} - пока List[Layer] для простоты, в будущем нужно continuum
        self.square_points = [] # list<layers>(type=square_point) - List[Layer]
        self.logistic_map_points = [] # list<layers>(type=logistic_map_point) - List[Layer]
        self.board_cells = [] # list<layers>(type=game_cell) - List[Layer]
        self.current_player = None # enums.Player - Enum Player
        self.game_state = None # enums.GameState - Enum GameState

    def __repr__(self): # Для удобного вывода объектов Figure
        return f"Figure(name='{self.name}', type='{self.type}', center={self.center.name if self.center else None}, radius={self.radius}, side_length={self.side_length}, ...)" # ... (можно добавить больше свойств для вывода)

In [3]:
class Player(Enum):
    X = "X"
    O = "O"
    NONE = "None" # Переименовано из None, так как None - ключевое слово Python

class CellState(Enum):
    EMPTY = "Empty"
    X_MARK = "X_Mark"
    O_MARK = "O_Mark"

class GameState(Enum):
    IN_PROGRESS = "InProgress" # Переименовано для PEP8
    X_WINS = "X_Wins" # Переименовано для PEP8
    O_WINS = "O_Wins" # Переименовано для PEP8
    DRAW = "Draw"

In [4]:
import math

def create_layer(name, layer_type, x=None, y=None, z=None, state=None, row=None, col=None, is_in_set=None, iteration_value=None):
    """
    Функция для создания экземпляра класса Layer.
    """
    return Layer(name=name, layer_type=layer_type, x=x, y=y, z=z, state=state, row=row, col=col, is_in_set=is_in_set, iteration_value=iteration_value)

def distance(layer1, layer2):
    """
    Функция для вычисления евклидова расстояния между двумя слоями (точками) в 3D пространстве.
    Предполагает, что layer1 и layer2 имеют атрибуты x, y, z.
    """
    if layer1.x is None or layer1.y is None or layer1.z is None or layer2.x is None or layer2.y is None or layer2.z is None:
        raise ValueError("Для вычисления расстояния координаты x, y, z должны быть определены для обоих слоев.")
    dx = layer1.x - layer2.x
    dy = layer1.y - layer2.y
    dz = layer1.z - layer2.z
    return math.sqrt(dx**2 + dy**2 + dz**2)

def radians(degrees):
    """
    Функция для перевода градусов в радианы.
    Использует math.radians из стандартной библиотеки math.
    """
    return math.radians(degrees)

def cos(angle_radians):
    """
    Функция для вычисления косинуса угла в радианах.
    Использует math.cos из стандартной библиотеки math.
    """
    return math.cos(angle_radians)

def sin(angle_radians):
    """
    Функция для вычисления синуса угла в радианах.
    Использует math.sin из стандартной библиотеки math.
    """
    return math.sin(angle_radians)

def assert_rule(condition, message):
    """
    Функция для обработки assert выражений в правилах.
    Если условие condition ложно, выбрасывает исключение AssertionError с заданным сообщением message.
    """
    if not condition:
        raise AssertionError(message)

# Для работы с комплексными числами (если потребуется, например, для множества Мандельброта)
def complex_num(real, imag):
    """
    Функция для создания комплексного числа.
    Использует built-in complex type in Python.
    """
    return complex(real, imag)

def abs_complex(complex_number):
    """
    Функция для вычисления модуля (абсолютного значения) комплексного числа.
    Использует built-in abs() function for complex numbers in Python.
    """
    return abs(complex_number)

In [5]:
import abstract_math_core
point1 = create_layer(name="PointA", layer_type=LayerType.POINT, x=0.0, y=0.0, z=0.0)
print(point1)
point2 = create_layer(name="PointB", layer_type=LayerType.POINT, x=3.0, y=4.0, z=0.0)
print(point2)
distance_points = distance(point1, point2)
print(f"Расстояние между {point1.name} и {point2.name}: {distance_points}")
circle1 = Figure(name="Circle1", figure_type="circle")
print(circle1)
assert_rule(True, "Это сообщение не должно появиться, так как условие True") # Не должно быть вывода
try:
    assert_rule(False, "Это сообщение должно появиться, так как условие False")
except AssertionError as e:
    print(f"AssertionError поймана: {e}")
    player_x = Player.X
print(f"Игрок: {player_x}, Значение: {player_x.value}")

cell_empty = CellState.EMPTY
print(f"Состояние клетки: {cell_empty}, Значение: {cell_empty.value}")

game_in_progress = GameState.IN_PROGRESS
print(f"Состояние игры: {game_in_progress}, Значение: {game_in_progress.value}")

Layer(name='PointA', type='point', x=0.0, y=0.0, z=0.0, state=None, row=None, col=None, is_in_set=None, iteration_value=None)
Layer(name='PointB', type='point', x=3.0, y=4.0, z=0.0, state=None, row=None, col=None, is_in_set=None, iteration_value=None)
Расстояние между PointA и PointB: 5.0
Figure(name='Circle1', type='circle', center=None, radius=None, side_length=None, ...)
AssertionError поймана: Это сообщение должно появиться, так как условие False
Игрок: Player.X, Значение: X
Состояние клетки: CellState.EMPTY, Значение: Empty
Состояние игры: GameState.IN_PROGRESS, Значение: InProgress


In [19]:
!python3 parse_rules.py parse_test.parse_rules


----- Парсинг файла: parse_test.parse_rules -----
Код из файла:
abstract_math define_elements(){layers; figures; enums}

enums define_enums(){
    TestEnum = enum{A, B, C}
}
Ошибка парсинга: некорректный формат значений enum для 'TestEnum' в блоке define_enums

Результат парсинга:
Парсинг не удался (вернул None)
